In [ ]:
import cv2
import mediapipe as mp
import math


mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as holistic:
    is_leftJab = False
    count = 0 
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        
        if results.face_landmarks:
            mp_drawing.draw_landmarks(
                image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style())

        
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, 
                results.pose_landmarks, 
                mp_holistic.POSE_CONNECTIONS, # To rysuje linie między barkami a łokciami
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())

        
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        
        
        if results.pose_landmarks:
            # 1. Pobieramy wymiary obrazu (potrzebne do przeliczenia na piksele)
            h, w, c = image.shape

    
            right_wrist = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_WRIST]
            left_wrist = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_WRIST]
            right_shoulder = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_SHOULDER]
            left_shoulder = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_SHOULDER]
            left_elbow = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_ELBOW]

    
            cx, cy, cz = right_wrist.x * w, right_wrist.y * h, right_wrist.z * w
            dx, dy, dz = left_wrist.x * w, left_wrist.y * h, left_wrist.z * w
            ex, ey, ez = right_shoulder.x * w, right_shoulder.y * h, right_shoulder.z * w
            fx, fy, fz = left_shoulder.x * w, left_shoulder.y * h, left_shoulder.z * w
            gx, gy, gz = left_elbow.x * w, left_elbow.y * h, left_elbow.z * w

    
            

            AC = math.sqrt((dx-fx)**2 + (dy-fy)**2)
            AB = math.sqrt((dx-gx)**2 + (dy-gy)**2)
            BC = math.sqrt((fx-gx)**2 + (fy-gy)**2)

            cosangle = (AB**2 + BC**2 - AC**2)/(2*AB*BC)
            #twierdzenie cos jest zbyt kosztowne
            #jednak nie

            
            
            angle = math.degrees(math.acos(cosangle))
               
            print(angle)
            if is_leftJab == False and angle >= 170:
                is_leftJab = True 
                count+=1
            if is_leftJab == True and angle < 170:
                is_leftJab = False
            



            #narazie celem jest stworzenie sesji treningowej, ktora bedzie mierzyla predkosc kazdego ciosu
            #lewego prostego

            
            








        


            

        cv2.imshow('MediaPipe Body Flow - Full Tracking', image)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            print(count)
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#roznica

In [ ]:
import cv2
import mediapipe as mp
import math
import time


mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

cap = cv2.VideoCapture(0)

# zdefiniowanie czasu
prev_time = 0
count1 = 0
prev_dx = 0
prev_dy = 0

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as holistic:
    is_leftJab = False
    count = 0 
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        
        if results.face_landmarks:
            mp_drawing.draw_landmarks(
                image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style())

        
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image, 
                results.pose_landmarks, 
                mp_holistic.POSE_CONNECTIONS, # To rysuje linie między barkami a łokciami
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())

        
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        
        
        if results.pose_landmarks:
            # 1. Pobieramy wymiary obrazu (potrzebne do przeliczenia na piksele)
            h, w, c = image.shape

    
            right_wrist = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_WRIST]
            left_wrist = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_WRIST]
            right_shoulder = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.RIGHT_SHOULDER]
            left_shoulder = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_SHOULDER]
            left_elbow = results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_ELBOW]

    
            cx, cy, cz = right_wrist.x * w, right_wrist.y * h, right_wrist.z * w
            dx, dy, dz = left_wrist.x * w, left_wrist.y * h, left_wrist.z * w
            ex, ey, ez = right_shoulder.x * w, right_shoulder.y * h, right_shoulder.z * w
            fx, fy, fz = left_shoulder.x * w, left_shoulder.y * h, left_shoulder.z * w
            gx, gy, gz = left_elbow.x * w, left_elbow.y * h, left_elbow.z * w

            '''if count1 == 0:
                prev_dx = dx
                prev_dy = dy
                count+=1'''

            
    
            #print(f"Prawy nadgarstek -> X: {cx}px, Y: {cy}px | (Z-depth: {right_wrist.z:.4f})")
            #print(f"Lewy nadgarstek -> X: {dx}px, Y: {dy}px | (Z-depth: {left_wrist.z:.4f})")

                # Rysuje tekst bezpośrednio na obrazie z kamery
            #cv2.putText(image, f"X:{cx} Y:{cy}", (cx, cy - 20), 
                   #cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
    # odleglosc bark-nadgarstek w 3d
            #C to bark
            #wektory
            vector_BAx, vector_BAy = left_wrist.x - left_elbow.x, left_wrist.y - left_elbow.y
            vector_BCx, vector_BCy = left_shoulder.x - left_elbow.x, left_shoulder.y - left_elbow.y 

            AC = math.sqrt((dx-fx)**2 + (dy-fy)**2)
            AB = math.sqrt((dx-gx)**2 + (dy-gy)**2)
            BC = math.sqrt((fx-gx)**2 + (fy-gy)**2)

            cosangle = (AB**2 + BC**2 - AC**2)/(2*AB*BC)
            #twierdzenie cos jest zbyt kosztowne
            #jednak nie

            dot_product = (vector_BAx * vector_BCx + vector_BAy*vector_BCy)
            
            angle = math.degrees(math.acos(cosangle))
               
            #print(angle)
            '''if is_leftJab == False and angle >= 170:
                is_leftJab = True 
                count+=1
                start_time = time.perf_counter()
            if is_leftJab == True and angle < 170:
                is_leftJab = False'''

            
            #tab = [None][None]




            #narazie celem jest stworzenie sesji treningowej, ktora bedzie mierzyla predkosc kazdego ciosu
            #lewego prostego

            
            #czas
            curr_time = time.perf_counter()
            dt = curr_time - prev_time

            fps = 1 / dt
            #print(fps)

            #predkosc nadgarstka w 2d w pikselach
            dist = math.sqrt((dx-prev_dx)**2 + (dy-prev_dy)**2)
            dspeed = dist / dt
            #print(dspeed)
            #print(dt)
            #print(dspeed)  
            
            if is_leftJab == False and dspeed >= 100:
                is_leftJab = True 
                count+=1
                print(count)
                print(dspeed)
                
            if is_leftJab == True and dspeed <= 20:
                is_leftJab = False
            print(count)


            #na koniec
            prev_time = curr_time
            prev_dx = dx
            prev_dy = dy








        


            

        cv2.imshow('MediaPipe Body Flow - Full Tracking', image)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            print(count)
            break

cap.release()
cv2.destroyAllWindows()

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
2
2
2
2
2
2
2
2
2
2
2
3
3
3
4
4
4
4
4
4
4
4
5
5
5
5
5
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
9
9
9
9
9
9
9
9
9
9
10
10
10
10
10
10
10
10
10
10
11
11
11
11
11
11
11
12
12
12
12
12
12
12
12
12
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
14
15
15
15
15
15
15
15
15
15
15
15
16
16
16
16
17
17
17
17
17
17
17
18
18
18
18
18
18
18
18
18
18
18
18
18
18
19
19
19
19
19
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
21
21
21
21
21
21
21
21
21
21
21
21
21
21
21
21
21
2